In [2]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

text = "Hi, how are you"
ids = tok.encode(text)
print(ids)
print(len(ids), "tokens")
for i in ids:
    print(f"{i:>7} -> {tok.decode(i)!r}")



pairs = [
    ("Машинне навчання — це цікаво", "Machine learning is interesting"),
    ("Привіт світ", "Hello world"),
]

for uk, en in pairs:
    print(f"{len(tok.encode(uk)):3d} токенів | {uk}")
    print(f"{len(tok.encode(en)):3d} токенів | {en}\n")


[13048, 11, 1246, 525, 498]
5 tokens
  13048 -> 'Hi'
     11 -> ','
   1246 -> ' how'
    525 -> ' are'
    498 -> ' you'
 17 токенів | Машинне навчання — це цікаво
  4 токенів | Machine learning is interesting

  7 токенів | Привіт світ
  2 токенів | Hello world



In [3]:
from sentence_transformers import SentenceTransformer


model = SentenceTransformer("intfloat/multilingual-e5-small")

sentences = ["кіт сидить на килимку", "кошеня лежить на підлозі", "курс долара виріс"]
emb = model.encode(sentences)

print(emb.shape)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3836.59it/s]


(3, 384)


In [5]:
import numpy as np

def cosine(a, b):
    return np.dot(a,b) / (np.linalg.norm(a) * np.linalg.norm(b))


print(cosine(emb[0], emb[1]))     # кіт / кошеня — високо
print(cosine(emb[0], emb[2]))

0.9227107
0.81203705


In [6]:
sentences = [
    "як приготувати борщ",
    "рецепт українського супу",
    "як налаштувати Docker",
    "встановлення контейнерів",
    "погода в Києві завтра",
]

emb = model.encode(sentences, normalize_embeddings=True)
sim = emb @ emb.T                      # якщо нормалізовані — це вже косинус

import pandas as pd
print(pd.DataFrame(sim.round(3),
                   index=[s[:20] for s in sentences],
                   columns=range(len(sentences))))

                          0      1      2      3      4
як приготувати борщ   1.000  0.866  0.823  0.802  0.789
рецепт українського   0.866  1.000  0.810  0.813  0.796
як налаштувати Docke  0.823  0.810  1.000  0.871  0.753
встановлення контейн  0.802  0.813  0.871  1.000  0.793
погода в Києві завтр  0.789  0.796  0.753  0.793  1.000


In [12]:
docs = [
    "Python — мова програмування з динамічною типізацією.",
    "Docker дозволяє запускати застосунки в контейнерах.",
    "Борщ готують з буряком, капустою і мʼясом.",
    "PyTorch — бібліотека для глибокого навчання.",
    "Вареники ліплять з тіста і начинки.",
    "Kubernetes керує оркестрацією контейнерів.",
]

doc_emb = model.encode(docs, normalize_embeddings=True)

def search(query, top_k=2):
    q_emb = model.encode([query], normalize_embeddings=True)[0]
    scores = doc_emb @ q_emb
    idx = np.argsort(scores)[::-1][:top_k]
    for i in idx:
        print(f"{scores[i]:.3f}  {docs[i]}")

search("як працювати з контейнерами")
print()
search("українська кухня")
print()
search("нейронні мережі")
print()
search("чим Docker відрізняється від Kubernetus")
print()
search("Як повязані борщ і контейнери")

0.869  Docker дозволяє запускати застосунки в контейнерах.
0.869  Kubernetes керує оркестрацією контейнерів.

0.839  Вареники ліплять з тіста і начинки.
0.831  Борщ готують з буряком, капустою і мʼясом.

0.846  PyTorch — бібліотека для глибокого навчання.
0.802  Kubernetes керує оркестрацією контейнерів.

0.878  Kubernetes керує оркестрацією контейнерів.
0.864  Docker дозволяє запускати застосунки в контейнерах.

0.860  Борщ готують з буряком, капустою і мʼясом.
0.847  Kubernetes керує оркестрацією контейнерів.


In [13]:
q = "query: як працювати з контейнерами"
d = ["passage: " + doc for doc in docs]

q_emb = model.encode([q], normalize_embeddings=True)[0]
d_emb = model.encode(d, normalize_embeddings=True)

scores = d_emb @ q_emb
print(np.argsort(scores)[::-1][:2])

[5 1]


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

texts = [
    "Чудовий сервіс, дуже задоволений", "Все сподобалось, рекомендую",
    "Швидка доставка, якісний товар", "Прекрасна якість за свої гроші",
    "Дуже приємний персонал", "Все ідеально, буду замовляти ще",
    "Жахливо, гроші на вітер", "Товар зламався через день",
    "Débilна доставка, чекав місяць", "Не рекомендую нікому",
    "Якість огидна, поверніть гроші", "Найгірший досвід у житті",
]
labels = [1]*6 + [0]*6

X = model.encode(texts, normalize_embeddings=True)
print(X.shape)        # (12, 384)

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.34, random_state=42, stratify=labels
)

clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
print(classification_report(y_test, clf.predict(X_test)))

(12, 384)
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.40      1.00      0.57         2

    accuracy                           0.40         5
   macro avg       0.20      0.50      0.29         5
weighted avg       0.16      0.40      0.23         5



/home/illidan/JuniorMLPrep/llm-playground/.venv/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/illidan/JuniorMLPrep/llm-playground/.venv/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/illidan/JuniorMLPrep/llm-playground/.venv/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

In [10]:
models = [
    "intfloat/multilingual-e5-small",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
]

pairs = [
    ("кіт сидить на килимку", "кошеня лежить на підлозі", "близько"),
    ("як налаштувати Docker", "встановлення контейнерів", "близько"),
    ("погода в Києві", "курс долара", "далеко"),
]

for name in models:
    m = SentenceTransformer(name)
    print(f"\n{name}  (dim={m.get_embedding_dimension()})")
    for a, b, expect in pairs:
        e = m.encode([a, b], normalize_embeddings=True)
        print(f"  {e[0] @ e[1]:.3f}  ({expect})  {a[:25]} / {b[:25]}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3751.95it/s]
/tmp/ipykernel_47932/2856725332.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"\n{name}  (dim={m.get_sentence_embedding_dimension()})")



intfloat/multilingual-e5-small  (dim=384)
  0.923  (близько)  кіт сидить на килимку / кошеня лежить на підлозі
  0.871  (близько)  як налаштувати Docker / встановлення контейнерів
  0.822  (далеко)  погода в Києві / курс долара


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4085.71it/s]
/tmp/ipykernel_47932/2856725332.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"\n{name}  (dim={m.get_sentence_embedding_dimension()})")



sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2  (dim=384)
  0.863  (близько)  кіт сидить на килимку / кошеня лежить на підлозі
  0.612  (близько)  як налаштувати Docker / встановлення контейнерів
  0.058  (далеко)  погода в Києві / курс долара


In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer

docs = [
    "Docker дозволяє запускати застосунки в ізольованих контейнерах.",
    "Борщ готують з буряком, капустою, картоплею і мʼясом.",
    "PyTorch — бібліотека для навчання нейронних мереж.",
    "Вареники ліплять з тіста і начинки, наприклад картоплі чи вишні.",
    "Kubernetes автоматично масштабує і перезапускає контейнери.",
    "Градієнтний спуск оновлює ваги в напрямку, протилежному градієнту.",
    "Київ — столиця України, розташована на Дніпрі.",
    "Курс гривні до долара залежить від попиту на валюту.",
]

queries = [
    ("як ізолювати застосунок від системи", 0),
    ("рецепт червоного супу з буряком", 1),
    ("фреймворк для глибокого навчання", 2),
    ("страва з тіста з вишнями", 3),
    ("оркестрація контейнерів у кластері", 4),
    ("як оптимізатор змінює параметри моделі", 5),
    ("яке місто стоїть на Дніпрі", 6),
    ("від чого залежить обмінний курс", 7),
]

models = [
    "intfloat/multilingual-e5-small",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
]

for name in models:
    m = SentenceTransformer(name)
    is_e5 = False
    q_texts = [("query: " if is_e5 else "") + q for q, _ in queries]
    d_texts = [("passage: " if is_e5 else "") + d for d in docs]

    qe = m.encode(q_texts, normalize_embeddings=True)
    de = m.encode(d_texts, normalize_embeddings=True)
    scores = qe @ de.T                         # (8 запитів, 8 документів)

    correct = np.array([c for _, c in queries])
    top1 = scores.argmax(axis=1)
    acc = (top1 == correct).mean()

    # відрив: score правильного мінус найкращий неправильний
    margins = []
    for i, c in enumerate(correct):
        wrong = np.delete(scores[i], c)
        margins.append(scores[i, c] - wrong.max())

    print(f"\n{name}")
    print(f"  accuracy@1: {acc:.2f}")
    print(f"  середній відрив: {np.mean(margins):.3f}")
    for i, (q, c) in enumerate(queries):
        mark = "✓" if top1[i] == c else "✗"
        print(f"  {mark} {q[:40]:40s} → {docs[top1[i]][:35]}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 757.59it/s]



intfloat/multilingual-e5-small
  accuracy@1: 0.88
  середній відрив: 0.040
  ✓ як ізолювати застосунок від системи      → Docker дозволяє запускати застосунк
  ✓ рецепт червоного супу з буряком          → Борщ готують з буряком, капустою, к
  ✓ фреймворк для глибокого навчання         → PyTorch — бібліотека для навчання н
  ✓ страва з тіста з вишнями                 → Вареники ліплять з тіста і начинки,
  ✗ оркестрація контейнерів у кластері       → Docker дозволяє запускати застосунк
  ✓ як оптимізатор змінює параметри моделі   → Градієнтний спуск оновлює ваги в на
  ✓ яке місто стоїть на Дніпрі               → Київ — столиця України, розташована
  ✓ від чого залежить обмінний курс          → Курс гривні до долара залежить від 


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 439.95it/s]



sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  accuracy@1: 1.00
  середній відрив: 0.170
  ✓ як ізолювати застосунок від системи      → Docker дозволяє запускати застосунк
  ✓ рецепт червоного супу з буряком          → Борщ готують з буряком, капустою, к
  ✓ фреймворк для глибокого навчання         → PyTorch — бібліотека для навчання н
  ✓ страва з тіста з вишнями                 → Вареники ліплять з тіста і начинки,
  ✓ оркестрація контейнерів у кластері       → Kubernetes автоматично масштабує і 
  ✓ як оптимізатор змінює параметри моделі   → Градієнтний спуск оновлює ваги в на
  ✓ яке місто стоїть на Дніпрі               → Київ — столиця України, розташована
  ✓ від чого залежить обмінний курс          → Курс гривні до долара залежить від 
